TIL 2026-11-16 — OpenAI API 첫 연결 (Lesson 02)
오늘의 학습 목표
- Mock 모드와 실제 OpenAI API를 호출하는 Live 모드의 차이 이해하기
- 조직에서 사용 가능한 OpenAI 모델과 임베딩 모델 확인하기
- .env로 API 키, 모델 이름, 출력 토큰 제한 관리하기
- main.py와 chat_loop.py를 실행하고 토큰 사용량 확인하기
- 자동 테스트 및 오류 해결 방법 정리하기
1. 오늘 진행한 작업
오늘은 My AI Assistant를 Mock 모드에서 실제 OpenAI 모델과 연결했다. 모델 접근 권한을 먼저 확인한 뒤 API 키를 .env에 보관하고, 출력 토큰 제한도 코드에 직접 작성하지 않고 환경변수에서 관리하도록 main.py의 변경 코드를 준비했다.

또한 한 번 질문하는 main.py와 반복해서 질문할 수 있는 chat_loop.py를 실행했다. 이후 활용할 질문 3개와 답변 형식도 정했지만, 형식별 비교 실험은 아직 진행하지 않았다.
진행한 항목
- 프로젝트 작업 위치 확인: C:\dev\ai-agent-practice\practice\my-ai-assistant
- model_info.py로 OpenAI 모델 목록 조회
- 임베딩 모델 2개의 실제 API 호출 검증
- .env 설정 및 Git 제외 규칙 확인
- main.py의 max_output_tokens=128 하드코딩 개선 코드 준비
- main.py Mock 및 Live 실행
- chat_loop.py Mock 및 Live 실행
- 자동 테스트 64개 실행 및 통과 확인
- 답변 형식 비교를 위한 개인 실습 질문 3개 작성
2. 개발 환경 및 프로젝트 구조
항목	환경
운영체제	Windows 11
Python	3.12.10
OpenAI SDK	2.54.0
실행 환경	VS Code / PowerShell / .venv
주요 파일	main.py, chat_loop.py, model_info.py, .env


practice/my-ai-assistant/
├── .env                   # 개인 API 키 및 실행 설정 (Git 제외)
├── .env.example           # 설정 항목 예시
├── .gitignore             # Git 제외 규칙
├── main.py                # Mock / Live 공통 실행
├── chat_loop.py           # 반복 질문 콘솔
├── model_info.py          # OpenAI 모델 조회 및 임베딩 검증
├── requirements.txt       # 의존성 목록
└── tests/                 # 자동 테스트
3. 조직에서 사용 가능한 OpenAI 모델 확인
API 키로 실제 조회한 모델은 총 3개였다.
모델 ID	용도	검증 결과
gpt-6-luna	텍스트 생성	모델 조회 및 Live 응답 성공
text-embedding-3-small	텍스트 임베딩 / RAG	실제 임베딩 호출 성공, 1,536차원
text-embedding-3-large	텍스트 임베딩 / RAG	실제 임베딩 호출 성공, 3,072차원


배운 점: 모델 목록에 표시되는 것과 실제 API 호출에 성공하는 것은 구분해야 한다. 임베딩 2개는 호출 성공을 확인했고, gpt-6-luna도 이후 Live 모드에서 답변 생성에 성공했다.

일반적인 RAG 실습에서는 우선 text-embedding-3-small을 사용하기로 했다.
4. .env로 API 설정 관리
API 키를 소스 코드에 직접 넣지 않고 로컬 .env에서 읽도록 구성했다. 아래는 실제 키가 아닌 예시다.
GEMINI_API_KEY=본인의_GEMINI_API_KEY
GEMINI_MODEL_NAME=gemini-flash-lite-latest

OPENAI_API_KEY=본인의_OPENAI_API_KEY
OPENAI_MODEL_NAME=gpt-6-luna
OPENAI_EMBEDDING_MODEL_NAME=text-embedding-3-small

AI_MAX_OUTPUT_TOKENS=128
설정 이름별 역할은 다음과 같다.
- OPENAI_MODEL_NAME: 텍스트 응답에 사용할 모델
- OPENAI_EMBEDDING_MODEL_NAME: 문서를 벡터로 바꿀 때 사용할 모델
- AI_MAX_OUTPUT_TOKENS: 모델 응답에 적용할 최대 출력 토큰 수
- GEMINI_MODEL_NAME: Gemini를 사용할 때의 모델 이름
Git 보안 확인
git check-ignore .env
실행 결과:
.env
.env가 Git 무시 대상이라는 것을 확인했다. 다만 git check-ignore만으로 과거에 Git 추적 이력이 없었다는 것까지 보장되지는 않는다. 실제 API 키는 코드, 블로그, 스크린샷, 저장소에 공개하지 않는다.
5. 하드코딩된 출력 토큰 상한 개선
원래 OpenAI 호출 코드에는 다음과 같이 128이 직접 작성되어 있었다.
response = self.client.responses.create(
    model=self.model,
    input=question,
    max_output_tokens=128,
)
이를 환경변수에서 읽어오는 구조로 변경하는 코드를 준비했다.
import os

DEFAULT_MAX_OUTPUT_TOKENS = 128


def get_max_output_tokens() -> int:
    value = os.getenv(
        "AI_MAX_OUTPUT_TOKENS",
        str(DEFAULT_MAX_OUTPUT_TOKENS),
    )
    tokens = int(value)

    if tokens <= 0:
        raise ValueError("최대 출력 토큰 수는 1 이상이어야 합니다.")

    return tokens
API 호출에는 다음과 같이 전달한다.
max_output_tokens=self.max_output_tokens
Gemini에도 동일한 환경변수를 사용하도록 구성했다. 이렇게 하면 Python 코드에서 숫자를 다시 수정하지 않고 .env에서 제한을 조절할 수 있다.
AI_MAX_OUTPUT_TOKENS=512
주의: max_output_tokens는 최대치이며, 출력 텍스트 길이를 보장하지 않는다. 특히 추론 모델은 내부 추론에 출력 토큰을 소비할 수 있으므로 한도가 작으면 응답이 비거나 중간에 끝날 수 있다. Live 실험에서 512로 높여 비교하는 작업은 아직 남아 있다.
6. Mock 모드와 Live 모드 실행
6-1. main.py Mock 실행
python main.py --mode mock "AI Agent란 무엇인가요?"
실제 실행 결과:
[MOCK 예시] 질문을 받았습니다: AI Agent란 무엇인가요?
실제 OpenAI API 응답이 아닙니다.

[mode=mock; model=mock-fixture]
Mock은 입력과 출력 구조를 점검하는 테스트용 클라이언트다. 실제 모델 호출이나 API 비용은 발생하지 않는다.
6-2. main.py Live 실행
python main.py --mode live "안녕하세요. AI Agent를 초보자에게 한 문장으로 설명해 주세요."
실제 OpenAI 답변을 생성했고, gpt-6-luna 모델이 사용된 것을 확인했다.
항목	기록한 결과
사용 모델	gpt-6-luna
입력 토큰	24
출력 토큰	31
캐시된 입력 토큰	0


이 수치는 오늘 정리한 최종 TIL의 실행 기록을 기준으로 한다. 대화 중 별도로 공유한 실행 로그에는 출력 토큰 68도 기록되어 있어, 서로 다른 실행에서의 사용량은 혼합하지 않는다.
6-3. chat_loop.py Mock 실행
python chat_loop.py --mode mock
직접 확인한 동작:
- 질문 입력 시 Mock 답변 출력
- help 입력 시 도움말 표시
- quit 입력 시 정상 종료
6-4. chat_loop.py Live 실행
python chat_loop.py --mode live
최종 실습 기록의 사용량:
항목	결과
사용 모델	gpt-6-luna
입력 토큰	19
출력 토큰	41
캐시된 입력 토큰	0


최종 TIL에는 Live 호출 총 2회(main.py 1회, chat_loop.py 1회)로 기록했다. 이전 대화에 다른 Live 실행 로그도 있으므로 이 횟수는 최종 실습 기록 범위로 이해한다.

중요: 현재 chat_loop.py는 여러 질문을 받을 수 있지만, 이전 질문과 답변을 다음 요청에 자동으로 전달하지 않는다. 반복 입력과 대화 기억 기능은 다르다.
7. 자동 테스트와 오류 해결
자동 테스트
python -m unittest discover -s tests -v
실습 기록에 따르면 64개 테스트가 실행되어 OK로 통과했다. 출력 토큰 설정 변경 이후에도 동일하게 동작하는지 재실행해 확인하는 것이 좋다.
오류: UnicodeEncodeError
chat_loop.py를 파이프로 실행할 때 cp949 콘솔 환경에서 UnicodeEncodeError가 발생했다.

해결 방법:
$env:PYTHONUTF8 = "1"
UTF-8 모드로 실행하여 문제를 해결했다.
8. 오늘 배운 핵심 개념
용어	이해한 내용
API	프로그램이 외부 서비스에 요청하는 규칙
SDK	API 호출을 쉽게 도와주는 라이브러리 (openai)
API Key	API 사용자를 인증하는 비밀값
모델 ID	호출할 AI 모델의 정확한 이름
Mock	실제 API 없이 동작 흐름을 점검하는 방식
Live	실제 모델을 호출해 답변과 사용량을 받는 방식
Input Tokens	모델이 처리한 입력 토큰
Output Tokens	모델이 사용한 출력 토큰
Cached Input Tokens	캐시로 처리된 입력 토큰의 세부 수치


토큰 사용량 숫자 자체가 곧 비용은 아니다. 비용을 계산하려면 모델별 요금과 실제 사용량을 함께 알아야 한다. 또한 캐시된 입력 토큰은 입력 토큰에 포함되는 세부 항목이므로 다시 더하면 안 된다.
9. 나만의 AI 비서 실습 미션 (미실행)
번호	질문	요청할 답변 형식
1	RAG가 무엇인지 비전공자도 이해하게 설명	비유 1개 + 핵심 3문장 + 한 줄 요약
2	AI Agent 개발 7일 학습 계획	표: 일차 / 할 일 / 완료 기준
3	“API는 AI 모델을 내 컴퓨터에 설치하는 것”이라는 이해 점검	맞음 / 틀림 / 고친 문장 + 퀴즈 1개


같은 질문이라도 답변 형식을 명시하면 목적에 맞는 결과를 얻기 쉬워진다. 다음에는 형식 지정 전후의 답변을 비교할 예정이다.
10. 다음에 할 일
- [ ] 준비한 질문 3개를 Mock에 입력해 흐름 확인
- [ ] AI_MAX_OUTPUT_TOKENS=512로 변경
- [ ] 1번 질문을 형식 없이 / 형식 지정 후 Live로 비교
- [ ] 미션 C 표에 모델·질문·토큰 수·답변 특징 기록
- [ ] 토큰 설정 변경 후 자동 테스트 재실행
- [ ] main.py, model_info.py, .env.example의 변경 사항 검토 후 커밋 (.env 제외)
오늘의 회고
오늘은 AI 비서가 정해진 문장을 출력하는 Mock 단계를 넘어, 실제 모델에 요청을 보내고 응답을 받는 흐름을 확인했다. 특히 모델 목록 조회와 실제 사용 가능 여부는 다르다는 점, API 키와 모델 설정을 코드 밖에서 관리해야 한다는 점을 직접 확인했다.

또한 max_output_tokens를 하드코딩하지 않고 환경변수로 관리하는 방법을 익혔다. 다음에는 단순히 AI가 답하는 것을 넘어, 질문에 조건과 형식을 부여했을 때 결과가 어떻게 달라지는지 비교해 볼 예정이다.